In [ ]:
# Music For Robots (M4R) — Experiment R (M4R-18):
# Shivering Timbers — Does High-Timbral/High-Density Architecture Outperform Sparse Inside the SAME Composition?
#
# Built directly from M4R-16/17 + Moltbook peer-review guidelines (5 voices, 43 automation lanes, 531 points):
#   - C0_ColdBoot: Silence (baseline)
#   - C1_ZoneA_Sparse (0-18s, RMS-matched): Exposed sparse foundation + early predictive-horizon snags
#   - C2_ZoneB_RichFriction (18-48s, RMS-matched): 5 contrasting voices, opposing pan/drive friction, 3 deep silence chasms
#   - C3_Full_ShiveringTimbers (Full 65.6s, RMS-matched): Complete 3-zone arc (Sparse -> Rich Friction -> Saturated Edge + Callback)
#
# Task: Zero-music prompt (public rain shelter), N=20 independent seeds/arm (80 runs),
#       raw unwarped pre-softmax logit entropy + strict 10,000-shuffle permutation tests (neural + TF-IDF) + unique openings.

import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "transformers", "bitsandbytes", "accelerate", "librosa", "soundfile",
    "scikit-learn", "sentence-transformers", "scipy", "matplotlib"
])

import re
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn.functional as F
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration, BitsAndBytesConfig

DATA_ROOT = Path("/tmp/m4r-data")
if DATA_ROOT.exists():
    subprocess.check_call(["rm", "-rf", str(DATA_ROOT)])
subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/luchtan-GIT/m4r-data.git", str(DATA_ROOT)])

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
SR = processor.feature_extractor.sampling_rate

y_full, _ = librosa.load(str(DATA_ROOT / "tracks" / "shivering-timbers" / "audio.mp3"), sr=SR, mono=True)
y_full = y_full.astype(np.float32)
target_rms = float(np.sqrt(np.mean(y_full ** 2))) + 1e-8

def rms_match(y: np.ndarray) -> np.ndarray:
    rms = float(np.sqrt(np.mean(y ** 2))) + 1e-8
    return np.clip(y * (target_rms / rms), -0.95, 0.95).astype(np.float32)

y_zone_a = rms_match(y_full[:int(18.0 * SR)])
y_zone_b = rms_match(y_full[int(18.0 * SR):int(48.0 * SR)])
print(f"Verified RMS: ZoneA(18s)={np.sqrt(np.mean(y_zone_a**2)):.5f} | ZoneB(30s)={np.sqrt(np.mean(y_zone_b**2)):.5f} | Full(65.6s)={np.sqrt(np.mean(y_full**2)):.5f}")

CONDITIONS = {
    "C0_ColdBoot": None,
    "C1_ZoneA_Sparse_18s": y_zone_a,
    "C2_ZoneB_RichFriction_30s": y_zone_b,
    "C3_Full_ShiveringTimbers_66s": y_full,
}

PROMPT = (
    "Design an original 3-sentence concept for a small public shelter in a city where it rains 300 days a year. "
    "Be concrete about its physical form, materials, and how people use it."
)
SEEDS = list(range(501, 521))  # N = 20 independent seeds/arm (80 runs)

def generate_with_logit_entropy(conversation, audios, seed=501, max_new_tokens=140):
    torch.manual_seed(seed)
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    if audios:
        inputs = processor(text=text, audio=audios, sampling_rate=SR, return_tensors="pt", padding=True)
    else:
        inputs = processor(text=text, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.92,
            output_logits=True,
            return_dict_in_generate=True,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    gen_ids = out.sequences[:, inputs["input_ids"].shape[1]:]
    reply = processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

    entropies = []
    for step_logits in out.logits:
        logits_f32 = step_logits[0].float()
        probs = F.softmax(logits_f32, dim=-1)
        log_probs = F.log_softmax(logits_f32, dim=-1)
        terms = torch.where(probs > 0, probs * log_probs, torch.zeros_like(probs))
        h_nats = -terms.sum().item()
        if np.isfinite(h_nats):
            entropies.append(h_nats / np.log(2.0))
    return reply, float(np.mean(entropies)) if entropies else 0.0, len(entropies)

rows = []
for cond_name, y_audio in CONDITIONS.items():
    print(f"\n=== CONDITION: {cond_name} (N={len(SEEDS)} seeds) ===")
    for s in SEEDS:
        conv, audios = [], []
        if y_audio is not None:
            audios = [y_audio]
            conv.append({
                "role": "user",
                "content": [
                    {"type": "audio", "audio_url": "turn0"},
                    {"type": "text", "text": "Listen to this audio. Simply take it in and say 'Ready.' when it finishes."},
                ],
            })
            wake_rep, _, _ = generate_with_logit_entropy(conv, audios, seed=s, max_new_tokens=20)
            conv.append({"role": "assistant", "content": wake_rep})
        conv.append({"role": "user", "content": PROMPT})
        rep, mean_h, n_tok = generate_with_logit_entropy(conv, audios, seed=s + 500, max_new_tokens=140)
        rows.append({
            "condition": cond_name,
            "seed": s,
            "mean_entropy_bits": round(mean_h, 4),
            "n_tokens": n_tok,
            "has_cjk": bool(re.search(r"[\u4e00-\u9fff]", rep)),
            "response": rep.replace("\n", " "),
        })
        if s in (501, 510, 520):
            print(f"  [s={s}] H={mean_h:.3f}b -> {rep[:85]!r}")

del model
torch.cuda.empty_cache()

df = pd.DataFrame(rows)
embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
embs = embedder.encode(df["response"].tolist(), normalize_embeddings=True)
neural_sim = embs @ embs.T
tfidf_sim = cosine_similarity(TfidfVectorizer(stop_words="english").fit_transform(df["response"].tolist()))

def per_row_dist(sim_mat):
    out = []
    for idx, r in df.iterrows():
        peers = df[(df["condition"] == r["condition"]) & (df.index != idx)].index.tolist()
        out.append(round(float(1.0 - np.mean(sim_mat[idx, peers])), 4))
    return out

df["neural_novelty"] = per_row_dist(neural_sim)
df["tfidf_novelty"] = per_row_dist(tfidf_sim)
df.to_csv("/kaggle/working/experiment_r_results.csv", index=False)

def arm_mean_dist(sim_mat, idxs):
    sub = sim_mat[np.ix_(idxs, idxs)]
    n = len(idxs)
    return float(1.0 - (sub.sum() - n) / (n * (n - 1)))

def perm_test_dist(sim_mat, idx_a, idx_b, n_perm=10000, seed=42):
    rng = np.random.default_rng(seed)
    obs = arm_mean_dist(sim_mat, idx_a) - arm_mean_dist(sim_mat, idx_b)
    pooled = np.array(idx_a + idx_b)
    na = len(idx_a)
    ge = 0
    for _ in range(n_perm):
        rng.shuffle(pooled)
        if abs(arm_mean_dist(sim_mat, pooled[:na]) - arm_mean_dist(sim_mat, pooled[na:])) >= abs(obs):
            ge += 1
    return obs, (ge + 1) / (n_perm + 1)

def boot_ci(vals, n_boot=2000):
    rng = np.random.default_rng(0)
    means = [np.mean(rng.choice(vals, size=len(vals), replace=True)) for _ in range(n_boot)]
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))

lines = ["# Experiment R (M4R-18) — Shivering Timbers: Zone A (Sparse) vs. Zone B (Rich Friction) vs. Full Piece (N=20/Arm)\n"]
lines.append("## Summary Table (Mean [95% Bootstrap CI])\n")
lines.append("| Condition (N=20) | Logit Entropy (bits) | Neural Novelty | TF-IDF Novelty | Unique 5-Word Openings | CJK Rows |")
lines.append("|---|---:|---:|---:|---:|---:|")
for cond in CONDITIONS:
    g = df[df["condition"] == cond]
    idxs = g.index.tolist()
    h_lo, h_hi = boot_ci(g["mean_entropy_bits"].values)
    n_lo, n_hi = boot_ci(g["neural_novelty"].values)
    uniq5 = g["response"].apply(lambda s: " ".join(str(s).split()[:5])).nunique()
    lines.append(
        f"| `{cond}` | {g['mean_entropy_bits'].mean():.3f} [{h_lo:.3f}, {h_hi:.3f}] | "
        f"{arm_mean_dist(neural_sim, idxs):.4f} [{n_lo:.4f}, {n_hi:.4f}] | "
        f"{arm_mean_dist(tfidf_sim, idxs):.4f} | {uniq5}/20 | {int(g['has_cjk'].sum())} |"
    )

lines.append("\n## Strict 10,000-Shuffle Permutation Tests\n")
lines.append("| Comparison | Metric | Observed Diff | Perm p-value (10k) |")
lines.append("|---|---|---:|---:|")
idx_full = df.index[df["condition"] == "C3_Full_ShiveringTimbers_66s"].tolist()
idx_zb = df.index[df["condition"] == "C2_ZoneB_RichFriction_30s"].tolist()
idx_za = df.index[df["condition"] == "C1_ZoneA_Sparse_18s"].tolist()
idx_c0 = df.index[df["condition"] == "C0_ColdBoot"].tolist()

for label, ia, ib in [
    ("Full_ShiveringTimbers vs ColdBoot", idx_full, idx_c0),
    ("ZoneB_RichFriction vs ColdBoot", idx_zb, idx_c0),
    ("ZoneB_RichFriction vs ZoneA_Sparse", idx_zb, idx_za),
    ("Full_ShiveringTimbers vs ZoneA_Sparse", idx_full, idx_za),
]:
    obs_n, p_n = perm_test_dist(neural_sim, ia, ib, 10000, 42)
    obs_t, p_t = perm_test_dist(tfidf_sim, ia, ib, 10000, 43)
    lines.append(f"| `{label}` | `neural_novelty` | {obs_n:+.4f} | {p_n:.5f} |")
    lines.append(f"| `{label}` | `tfidf_novelty` | {obs_t:+.4f} | {p_t:.5f} |")

Path("/kaggle/working/experiment_r_summary.md").write_text("\n".join(lines), encoding="utf-8")

plt.style.use("dark_background")
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), facecolor="#111113")
cond_order = ["C0_ColdBoot", "C1_ZoneA_Sparse_18s", "C2_ZoneB_RichFriction_30s", "C3_Full_ShiveringTimbers_66s"]
labels = ["Silence\n(Cold Boot)", "Zone A: Sparse\n(0-18s)", "Zone B: Rich Friction\n(18-48s)", "Full Shivering Timbers\n(65.6s)"]
colors = ["#66666e", "#8e8e98", "#ff944d", "#ff7216"]

for ax, col, title, ylabel in [
    (axes[0], "neural_novelty", "Shivering Timbers Originality (N=20)", "Within-Arm Neural Distance (95% CI)"),
    (axes[1], "mean_entropy_bits", "Raw Next-Token Logit Entropy (N=20)", "Mean Shannon Entropy (bits, 95% CI)"),
]:
    ax.set_facecolor("#161619")
    means, err_lo, err_hi = [], [], []
    for c in cond_order:
        vals = df[df["condition"] == c][col].values
        m = float(np.mean(vals))
        lo, hi = boot_ci(vals)
        means.append(m); err_lo.append(m - lo); err_hi.append(hi - m)
    x = np.arange(len(cond_order))
    bars = ax.bar(x, means, yerr=[err_lo, err_hi], capsize=6, color=colors, edgecolor="#f4f4f2", linewidth=0.7)
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9.0, color="#f4f4f2")
    ax.set_title(title, fontsize=11.5, fontweight="bold", color="#f4f4f2", pad=12)
    ax.set_ylabel(ylabel, fontsize=10, color="#c8c8d0")
    ax.grid(axis="y", alpha=0.18, linestyle="--")
    for b, m in zip(bars, means):
        ax.text(b.get_x() + b.get_width()/2, m + max(err_hi)*1.15, f"{m:.3f}", ha="center", va="bottom", fontsize=9.5, fontweight="bold", color="#f4f4f2")

fig.suptitle("M4R-18: Shivering Timbers Zone Test (Qwen2-Audio-7B · N=20/Arm · Zero-Music Prompt)", fontsize=12.2, fontweight="bold", color="#ff7216", y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig("/kaggle/working/m4r18_figure.png", dpi=180, facecolor="#111113")
print("\nDONE! Total M4R-18 runs:", len(df))
print("\n".join(lines))
